## MDCKII label preprocessing

Restart the kernel before running this code

In [ ]:
1+1

In [ ]:
import tifffile as tiff
import numpy as np
import pandas as pd
import napari
import matplotlib.pyplot as plt
from scipy.ndimage import find_objects, binary_closing 
from scipy.ndimage import label as nd_label
from skimage.measure import regionprops
from skimage.morphology import disk
from magicgui import magicgui

### Image directory

In [ ]:
route = ("../26.02.10_MDCKII/masks_intersect/mask_intersect_2.tif")

In [ ]:
labels = tiff.imread(route)  # shape: (Z, Y, X)

print(f"Shape:     {labels.shape}")
print(f"Nº cells: {len(np.unique(labels)) - 1}")
#print(f"Nº cells: {labels .max()}") #some labels may be missing due to intersection
print(f"Dtype:     {labels.dtype}")
print(f"Memory:   {labels.nbytes / 1e6:.1f} MB")

### Remove clearly artefactual labels

Floating debris or clear segmentation errors

In [ ]:
# Function to remove labels by clicking with Ctrl
viewer = napari.Viewer()
viewer.add_labels(labels, name='labels original')

removed_labels = []

@viewer.mouse_drag_callbacks.append
def remove_label_on_click(viewer, event):
    if 'Control' not in event.modifiers:
        return
    
    # Obtain cursor position and convert to integer indices
    coords = np.round(viewer.cursor.position).astype(int)
    z, y, x = coords[0], coords[1], coords[2]
    
    # Check if the coordinates are within bounds
    if not (0 <= z < labels.shape[0] and
            0 <= y < labels.shape[1] and
            0 <= x < labels.shape[2]):
        return
    
    lab = labels[z, y, x]
    if lab == 0:
        return
    
    # Remove label from the layer
    layer = viewer.layers['labels original']
    layer.data[layer.data == lab] = 0
    layer.refresh()
    
    removed_labels.append(lab)
    print(f"Label {lab} removed | Total removed: {len(removed_labels)}: {removed_labels}")

# Widget to apply removals and update labels variable
@magicgui(call_button='Apply removals')
def apply_removals():
    global labels
    layer = viewer.layers['labels original']
    labels = layer.data.copy()
    print(f"labels updated. Labels removed: {removed_labels}")
    print(f"Cells remaining: {len(np.unique(labels)) - 1}")

@magicgui(
    call_button='Remove by label',
    label_id={'label': 'Label'}
)

# Function to remove labels by entering label ID
def remove_by_label(label_id: int):
    layer = viewer.layers['labels original']

    if label_id == 0:
        print("The label 0 is the background and cannot be removed.")
        return

    if not np.any(layer.data == label_id):
        print(f"Label {label_id} not found.")
        return

    layer.data[layer.data == label_id] = 0
    layer.refresh()

    removed_labels.append(label_id)

    print(
        f"Label {label_id} removed | "
        f"Total removed: {len(removed_labels)}"
    )

viewer.window.add_dock_widget(apply_removals, area='right')
viewer.window.add_dock_widget(remove_by_label, area='right')

print("Ctrl + Click on a cell to remove it")
napari.run()

### Fusion overlapping cells

Avoid Z fragmentation of labels on monolayer cultures

In [ ]:
#Fusion labels with high overlap in YX footprint (any Z). Both cells must have a high fraction of their footprint overlapping to be fused.

#Footprint and YX bounding box for each label
print("Calculating footprints...")
footprints = {}  # label -> 2D YX mask (bool)
bboxes_yx  = {}  # label -> (y0, y1, x0, x1)

slices_3d = find_objects(labels)

for lab, bb in enumerate(slices_3d, start=1):
    if bb is None:
        continue
    # Footprint: YX projection (any occupied Z)
    crop      = labels[bb]
    fp        = (crop == lab).any(axis=0)  # shape (Y_crop, X_crop)
    footprints[lab] = fp

    # YX bounding box in global coordinates
    y0 = bb[1].start; y1 = bb[1].stop
    x0 = bb[2].start; x1 = bb[2].stop
    bboxes_yx[lab] = (y0, y1, x0, x1)

print(f"Calculated footprints: {len(footprints)}")

# Bidirectional overlap between bounding boxes
overlap_threshold = 0.60  # threshold for minimum bidirectional overlap to fuse

fusions = {}  # lab_src -> lab_dst

labs = list(footprints.keys())

for i, lab_a in enumerate(labs):
    y0a, y1a, x0a, x1a = bboxes_yx[lab_a]
    area_a = footprints[lab_a].sum()

    for lab_b in labs[i+1:]:
        y0b, y1b, x0b, x1b = bboxes_yx[lab_b]

        # Bounding box intersection in YX
        iy0 = max(y0a, y0b); iy1 = min(y1a, y1b)
        ix0 = max(x0a, x0b); ix1 = min(x1a, x1b)

        if iy0 >= iy1 or ix0 >= ix1:
            continue  # no overlap in bounding boxes

        # Overlap of footprints in the intersection region
        fp_a_crop = footprints[lab_a][iy0-y0a:iy1-y0a, ix0-x0a:ix1-x0a]
        fp_b_crop = footprints[lab_b][iy0-y0b:iy1-y0b, ix0-x0b:ix1-x0b]

        overlap   = int((fp_a_crop & fp_b_crop).sum())
        if overlap == 0:
            continue

        area_b    = footprints[lab_b].sum()

        # Bidirectional overlap: fraction with respect to each label
        pct_a = overlap / area_a  # A footprint fraction of the overlapping region
        pct_b = overlap / area_b  # B footprint fraction of the overlapping region

        # Bidirectional: both fractions must exceed the threshold
        if pct_a >= overlap_threshold and pct_b >= overlap_threshold:
            # Fusion the smaller label into the larger one (by volume)
            vol_a = (labels == lab_a).sum()
            vol_b = (labels == lab_b).sum()
            if vol_a >= vol_b:
                fusions[lab_b] = lab_a
            else:
                fusions[lab_a] = lab_b
            print(f"Fusion: {lab_b if vol_a >= vol_b else lab_a} -> "
                  f"{lab_a if vol_a >= vol_b else lab_b} "
                  f"(overlap {pct_a:.1%} / {pct_b:.1%})")

print(f"\nFusions detected: {len(fusions)}")

# Apply fusions
labels_fused = labels.copy()
for lab_src, lab_dst in fusions.items():
    labels_fused[labels_fused == lab_src] = lab_dst

print(f"Labels before:  {len(np.unique(labels))  - 1}")
print(f"Labels after: {len(np.unique(labels_fused)) - 1}")

In [ ]:
# Visualize original and fused labels with napari, highlighting the affected labels
viewer = napari.Viewer()

# Original image
viewer.add_labels(labels, name='Original', opacity=0.7)

# Fused image
viewer.add_labels(labels_fused, name='Fusionada', opacity=0.7)

# Mask of the labels that have been fused (src and dst)
labs_afectadas = set(fusions.keys()) | set(fusions.values())
mask_afectadas = np.isin(labels, list(labs_afectadas)).astype(np.float32)
viewer.add_image(mask_afectadas, name='Labels fusionadas', colormap='red',
                 opacity=1, blending='additive')

# Print summary in console for reference
print(f"Fusiones aplicadas: {len(fusions)}")
for src, dst in fusions.items():
    print(f"  {src} -> {dst}")

napari.run()

### Fusion XY fragmented cells

Some cells are fragmented on the XY plane

In [ ]:
labels_fused_XY = labels_fused.copy()

viewer = napari.Viewer()
viewer.add_labels(labels_fused_XY, name='labels_fused_XY')

selected_labels = []

@viewer.mouse_drag_callbacks.append
def select_label_on_click(viewer, event):
    if 'Control' not in event.modifiers:
        return
    coords = np.round(viewer.cursor.position).astype(int)
    z, y, x = coords[0], coords[1], coords[2]
    if not (0 <= z < labels_fused_XY.shape[0] and
            0 <= y < labels_fused_XY.shape[1] and
            0 <= x < labels_fused_XY.shape[2]):
        return
    lab = labels_fused_XY[z, y, x]
    if lab == 0:
        return
    if lab not in selected_labels:
        selected_labels.append(lab)
        print(f"Label selected: {lab} | Selected labels: {selected_labels}")
    else:
        print(f"Label {lab} already selected | Selected labels: {selected_labels}")

@magicgui(call_button='Fusion selected labels')
def fusion_labels():
    if len(selected_labels) < 2:
        print("Select at least 2 labels with Ctrl + Click")
        return

    # Fusion all labels in the first selected one
    lab_dst = selected_labels[0]
    layer   = viewer.layers['labels_fused_XY']

    for lab_src in selected_labels[1:]:
        layer.data[layer.data == lab_src] = lab_dst
        print(f"Label {lab_src} fusioned in {lab_dst}")

    layer.refresh()
    print(f"Fusion completed: {selected_labels} -> {lab_dst}")
    selected_labels.clear()

@magicgui(call_button='Clear selection')
def clear_selection():
    selected_labels.clear()
    print("Selection cleared")

@magicgui(call_button='Apply changes')
def apply_changes():
    global labels_fused_XY
    labels_fused_XY = viewer.layers['labels_fused_XY'].data.copy()
    print(f"labels_fused_XY updated. Cells: {len(np.unique(labels_fused_XY)) - 1}")

@magicgui(
    call_button='Fusion by IDs',
    labels='1,5,8'
)
def fusion_by_ids(labels: str = ''):
    """
    Introduce labels separated by commas.
    The first one will be the destination label.
    Example: 12,45,78 -> fuses 45 and 78 into 12
    """
    try:
        labs = [int(x.strip()) for x in labels.split(',') if x.strip()]
    except ValueError:
        print("Error: introduce integers separated by commas")
        return
    if len(labs) < 2:
        print("Introduce at least 2 labels")
        return
    layer = viewer.layers['labels_fused_XY']
    # Check that they exist
    presentes = np.unique(layer.data)
    faltantes = [lab for lab in labs if lab not in presentes]
    if faltantes:
        print(f"Labels not found: {faltantes}")
        return
    lab_dst = labs[0]
    for lab_src in labs[1:]:
        layer.data[layer.data == lab_src] = lab_dst
        print(f"Label {lab_src} fusioned in {lab_dst}")
    layer.refresh()
    print(f"Fusion completed: {labs} -> {lab_dst}")

viewer.window.add_dock_widget(fusion_labels,    area='right')
viewer.window.add_dock_widget(clear_selection,  area='right')
viewer.window.add_dock_widget(apply_changes,    area='right')
viewer.window.add_dock_widget(fusion_by_ids, area='right')

print("Ctrl + Click to select labels, then press 'Fusion selected labels' to fuse them.")
napari.run()

In [ ]:
# Backup of the fused labels during manual fusion, in case we want to save changes or revert

labels_fused_XY_backup = labels_fused_XY.copy()
#labels_fused_XY = labels_fused_XY_backup.copy()

### Filter labels by size

In [ ]:
# Analyze the resulting fused labels
rows = []
for r in regionprops(labels_fused_XY):
    vol = r.area

    rows.append({
        'label'       : r.label,
        'volume'      : vol,
    })

df = pd.DataFrame(rows)
print(f"Cells processed: {len(df)}")
print(df.drop(columns=['label']).describe())

In [ ]:
# Histogram of sizes before filtering
plt.figure(figsize=(8, 5))
plt.hist(df['volume'] / 1000, bins=100, color='skyblue', edgecolor='black')
plt.title('Distribution of Cellular Volumes')
plt.xlabel('Volume (×10³ voxels)')
plt.ylabel('Frequency')
plt.grid(axis='x', alpha=0.75)
plt.xticks(np.arange(0, (df['volume'].max() / 1000) + 10, 10))
plt.show()

In [ ]:
#Filter by size
df_filt = df[df['volume']>5000]

print(f"Cells preserved: {len(df_filt)}")
print(df_filt.drop(columns=['label']).describe())

In [ ]:
# Histogram of sizes before and after filtering
plt.figure(figsize=(8, 5))
plt.hist(df['volume']/1000, bins=100, alpha=0.5, label='All')
plt.hist(df_filt['volume']/1000, bins=100, alpha=0.5, label='Filtered')
plt.xticks(np.arange(0, (df['volume'].max() / 1000) + 10, 10))
plt.xlabel('Volume (×10³ voxels)')
plt.ylabel('Number of cells')
plt.title('Distribution of Cellular Sizes')  

In [ ]:
# Only labels in df_filt will be preserved
labels_ok = set(df_filt['label'].values)
labels_filt = np.where(np.isin(labels_fused_XY, list(labels_ok)), labels_fused_XY, 0).astype(labels_fused_XY.dtype)
print(f"Cells preserved after filtering: {len(np.unique(labels_filt)) - 1}")

In [ ]:
keep_map = np.vectorize(lambda x: 1 if x in labels_ok else 0)(labels_fused_XY).astype(np.uint8)
viewer = napari.Viewer()
viewer.add_labels(labels_fused_XY, name='fusioned')
viewer.add_labels(labels_filt, name='filtered by size')
viewer.add_labels(keep_map, name='preserved after filtering', opacity=0.9)
napari.run()

### Label closing

In [ ]:
# Close labels, to fill holes and improve cell segmentation.
# This is done separately for each label, to avoid merging adjacent cells.
# A 2D disk is used as the structuring element for closing, which can be adjusted in size.

struct = disk(11) # closing structure, with radius of 11 pixels
pad = 11  # >= radius of the structuring element, to ensure it fits within the cropped region around each label

labels_closed = np.zeros_like(labels_filt)

for z in range(labels_filt.shape[0]):
    print(f"Z-slide {z+1}")
    slice_ = labels_filt[z]
    out = np.zeros_like(slice_)

    # find_objects returns slices of the bounding box of each label
    slices = find_objects(slice_)

    for lab, bb in enumerate(slices, start=1):
        if bb is None:
            continue  # label ausente en este slice

        # Crop with padding, respecting borders
        y0 = max(bb[0].start - pad, 0)
        y1 = min(bb[0].stop  + pad, slice_.shape[0])
        x0 = max(bb[1].start - pad, 0)
        x1 = min(bb[1].stop  + pad, slice_.shape[1])

        crop = slice_[y0:y1, x0:x1]
        obj  = crop == lab
        obj_closed = binary_closing(obj, structure=struct)

        # Only fill the holes that were in the original object, to avoid merging adjacent cells
        region = out[y0:y1, x0:x1]
        region[obj_closed & (region == 0)] = lab

    labels_closed[z] = out

print(f"Shape:     {labels_closed.shape}")
print(f"Nº cells: {len(np.unique(labels_closed)) - 1}")
print(f"Dtype:     {labels_closed.dtype}")
print(f"Memory:   {labels_closed.nbytes / 1e6:.1f} MB")

### Manual corrections

In [ ]:
# Insert manual corrections if needed (e.g. filling holes between labels)
viewer = napari.Viewer()
label = 752 # visualize a specific label
mask_cell = (labels_closed == label)
viewer.add_labels(labels_closed, name='Labels', opacity=0.5)
viewer.add_image(mask_cell.astype(float), name=f'Cell {label}', colormap='magenta', blending='additive')
napari.run()

### Label 3D connectiviy

In [ ]:
# Keep only the largest connected component for each label in a 3D volume

def keep_largest_component_3d(labels: np.ndarray) -> np.ndarray:

    labels_out = labels.copy()
    slices = find_objects(labels)
    n_fragmentados = 0

    for lab, bb in enumerate(slices, start=1):
        if bb is None:
            continue

        crop = labels[bb]          # bounding box crop
        mask = crop == lab         # binary mask only for this label

        labeled_crop, n = nd_label(mask, structure=np.ones((3, 3, 3))) #neighborhood in cube 3x3 (26-neighborhood)
        if n <= 1:
            continue

        # Biggest fragment is the one with more pixels in the crop
        sizes = np.bincount(labeled_crop.ravel())
        sizes[0] = 0
        largest = sizes.argmax()

        # Remove fragments that are not the largest, by setting them to 0 in the output labels
        removed_mask = mask & (labeled_crop != largest)
        labels_out[bb][removed_mask] = 0

        removed = removed_mask.sum()
        n_fragmentados += 1
        print(f"Label {lab}: {n-1} fragments removed ({removed} vx)")

    print(f"\nLabels with fragments: {n_fragmentados}")
    return labels_out


labels_preprocessed = keep_largest_component_3d(labels_closed)

print(f"Shape:      {labels_preprocessed.shape}")
print(f"Nº cells: {len(np.unique(labels_preprocessed)) - 1}")
print(f"Dtype:      {labels_preprocessed.dtype}")
print(f"Memory:    {labels_preprocessed.nbytes / 1e6:.1f} MB")

### Remove cells from the borders


In [ ]:
# Remove cells whose bounding box touches the margin of `margin` pixels from any edge of the volume (Z, Y, X).
# These cells may be incomplete and bias the analysis

def remove_border_labels(labels: np.ndarray, marginxy: int = 11, marginz: int = 0) -> np.ndarray:

    Z, Y, X = labels.shape
    slices = find_objects(labels)
    border_labels = []

    for lab, bb in enumerate(slices, start=1):
        if bb is None:
            continue
        z0, z1 = bb[0].start, bb[0].stop
        y0, y1 = bb[1].start, bb[1].stop
        x0, x1 = bb[2].start, bb[2].stop

        if (z0 < marginz or z1 > Z - marginz or
            y0 < marginxy or y1 > Y - marginxy or
            x0 < marginxy or x1 > X - marginxy):
            border_labels.append(lab)

    print(f"Cells removed (border XY {marginxy}px, Z {marginz}px): {len(border_labels)}")

    labels_interior = labels.copy()
    labels_interior[np.isin(labels_interior, border_labels)] = 0
    return labels_interior

labels_noborder = remove_border_labels(labels_closed, marginxy=12, marginz=0) # marginxy matches the radius of the struct used in the closing (+1)
print(f"Shape:     {labels_noborder.shape}")
print(f"Nº cells: {len(np.unique(labels_noborder)) - 1}")
print(f"Dtype:     {labels_noborder.dtype}")
print(f"Memory:   {labels_noborder.nbytes / 1e6:.1f} MB")

### Save preprocessed images

Save border and borderless preprocessed images

In [ ]:
# guardar resultado final
route_out = route.replace(".tif", "_preprocessed.tif")
tiff.imwrite(route_out, labels_closed)

route_out = route.replace(".tif", "_noborder.tif")
tiff.imwrite(route_out, labels_noborder)